<a href="https://colab.research.google.com/github/prdip01/-cognify_task01/blob/main/Projectoneda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
np.random.seed(42)

n = 5000
df = pd.DataFrame({
    'invoice_no': [f'INV_{i}' for i in range(n)],
    'product': np.random.choice(['Laptop', 'Phone', 'Headphones', 'Charger', 'Case'], n),
    'quantity': np.random.randint(1, 5, n),
    'unit_price': np.random.choice([50, 100, 500, 1000, 50], n),
    'discount_percent': np.random.choice([0, 5, 10, 20, 30, 50], n, p=[0.2, 0.2, 0.2, 0.2, 0.15, 0.05]),
    'date': pd.date_range('2023-10-01', periods=n, freq='H')
})
df['revenue'] = df['quantity'] * df['unit_price'] * (1 - df['discount_percent']/100)
df['cost'] = df['unit_price'] * 0.6 * df['quantity']
df['profit'] = df['revenue'] - df['cost']
df.to_csv('discount_data.csv', index=False)

/tmp/ipykernel_4395/863989245.py:12: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  'date': pd.date_range('2023-10-01', periods=n, freq='H')


In [7]:

"""
RETAIL PRICING STRATEGY ANALYSIS
MBA-Level Discount Effectiveness Study
=====================================
Objective: Optimize profit margins by analyzing discount impact across products
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 1. LOAD AND PREPARE DATA
# ============================================================================
df = pd.read_csv('discount_data.csv')

print("=" * 90)
print("RETAIL PRICING STRATEGY ANALYSIS")
print("=" * 90)
print(f"\nDataset Shape: {df.shape}")
print(f"Products Analyzed: {df['product'].nunique()}")
print(f"Discount Levels: {sorted(df['discount_percent'].unique())}")

# ============================================================================
# 2. PROFIT MARGIN ANALYSIS BY DISCOUNT LEVEL
# ============================================================================

print("\n" + "=" * 90)
print("ANALYSIS 1: PROFIT MARGIN AT EACH DISCOUNT LEVEL")
print("=" * 90)

# Calculate metrics by discount level
discount_analysis = df.groupby('discount_percent').agg({
    'revenue': ['sum', 'mean'],
    'profit': ['sum', 'mean', 'count'],
    'cost': 'sum',
    'unit_price': 'mean',
    'quantity': 'sum'
}).round(2)

# Flatten multi-level column names
discount_analysis.columns = ['_'.join(col).strip() for col in discount_analysis.columns.values]

# Calculate profit margin
margin_by_discount = []
for discount in sorted(df['discount_percent'].unique()):
    subset = df[df['discount_percent'] == discount]
    total_revenue = subset['revenue'].sum()
    total_profit = subset['profit'].sum()
    total_cost = subset['cost'].sum() * subset['quantity'].sum()

    if total_revenue > 0:
        profit_margin = (total_profit / total_revenue) * 100
    else:
        profit_margin = 0

    margin_by_discount.append({
        'Discount %': f"{discount}%",
        'Total Revenue': f"${total_revenue:,.2f}",
        'Total Profit': f"${total_profit:,.2f}",
        'Profit Margin %': f"{profit_margin:.2f}%",
        'Avg Item Price': f"${subset['unit_price'].mean():,.2f}",
        'Transaction Count': len(subset)
    })

margin_df = pd.DataFrame(margin_by_discount)
print("\n", margin_df.to_string(index=False))

# ============================================================================
# 3. IDENTIFY SWEET SPOT (HIGHEST TOTAL PROFIT)
# ============================================================================

print("\n" + "=" * 90)
print("ANALYSIS 2: SWEET SPOT IDENTIFICATION (MAXIMUM PROFIT DISCOUNT)")
print("=" * 90)

profit_by_discount = df.groupby('discount_percent')['profit'].sum().sort_values(ascending=False)

sweet_spot_discount = profit_by_discount.idxmax()
sweet_spot_profit = profit_by_discount.max()
sweet_spot_revenue = df[df['discount_percent'] == sweet_spot_discount]['revenue'].sum()

print(f"\n✓ OPTIMAL DISCOUNT LEVEL: {sweet_spot_discount}%")
print(f"  • Total Profit Generated: ${sweet_spot_profit:,.2f}")
print(f"  • Total Revenue: ${sweet_spot_revenue:,.2f}")
print(f"  • Transactions: {len(df[df['discount_percent'] == sweet_spot_discount])}")
print(f"  • Profit vs 0% Discount: ${sweet_spot_profit - profit_by_discount[0]:,.2f}")

print("\n📊 Profit by Discount Level (Ranked):")
for discount, profit in profit_by_discount.items():
    pct_change = ((profit - profit_by_discount[0]) / profit_by_discount[0] * 100)
    print(f"   {discount}% Discount: ${profit:>12,.2f}  ({pct_change:>6.1f}%)")

# ============================================================================
# 4. CATEGORY-WISE ANALYSIS: PRODUCTS THAT SHOULD NEVER BE DISCOUNTED
# ============================================================================

print("\n" + "=" * 90)
print("ANALYSIS 3: CATEGORY-WISE DISCOUNT IMPACT")
print("=" * 90)

# For each product, find discount level with best profit margin
product_discount_impact = []

for product in df['product'].unique():
    product_df = df[df['product'] == product]

    best_discount = None
    best_profit_margin = -float('inf')
    worst_discount = None
    worst_profit_margin = float('inf')

    discount_stats = []

    for discount in sorted(product_df['discount_percent'].unique()):
        subset = product_df[product_df['discount_percent'] == discount]
        total_revenue = subset['revenue'].sum()
        total_profit = subset['profit'].sum()

        if total_revenue > 0:
            profit_margin = (total_profit / total_revenue) * 100
        else:
            profit_margin = 0

        discount_stats.append({
            'discount': discount,
            'profit': total_profit,
            'margin': profit_margin,
            'count': len(subset)
        })

        if profit_margin > best_profit_margin:
            best_profit_margin = profit_margin
            best_discount = discount

        if profit_margin < worst_profit_margin:
            worst_profit_margin = profit_margin
            worst_discount = discount

    product_discount_impact.append({
        'Product': product,
        'Best_Discount': best_discount,
        'Best_Margin': round(best_profit_margin, 2),
        'Worst_Discount': worst_discount,
        'Worst_Margin': round(worst_profit_margin, 2),
        'Margin_Gap': round(best_profit_margin - worst_profit_margin, 2),
        'Total_Profit': round(product_df['profit'].sum(), 2)
    })

product_impact_df = pd.DataFrame(product_discount_impact)
product_impact_df = product_impact_df.sort_values('Total_Profit', ascending=False)

print("\nProduct Performance Across Discount Levels:\n")
print(product_impact_df[['Product', 'Best_Discount', 'Best_Margin',
                         'Worst_Discount', 'Worst_Margin', 'Margin_Gap']].to_string(index=False))

# Identify high-margin products that suffer from discounts
print("\n" + "─" * 90)
print("🔴 PRODUCTS THAT SHOULD NEVER BE DISCOUNTED:")
print("─" * 90)

no_discount_products = product_impact_df[product_impact_df['Best_Discount'] == 0]
if len(no_discount_products) > 0:
    print("\nHighest margins at 0% discount (avoid discounting):")
    for _, row in no_discount_products.iterrows():
        print(f"  • {row['Product']:20s} | Best Margin: {row['Best_Margin']:6.2f}% @ 0% | "
              f"Margin Loss: {row['Margin_Gap']:6.2f}% if discounted")
else:
    print("No products exclusively best at 0% discount level")

# Identify high-risk discount products
high_risk = product_impact_df[product_impact_df['Margin_Gap'] > 20].sort_values('Margin_Gap', ascending=False)
if len(high_risk) > 0:
    print("\n⚠️  HIGH MARGIN SENSITIVITY (>20% gap between best & worst discount):")
    for _, row in high_risk.iterrows():
        print(f"  • {row['Product']:20s} | Gap: {row['Margin_Gap']:6.2f}% | "
              f"Max Loss at {row['Worst_Discount']}% discount")

# ============================================================================
# 5. REVENUE VS PROFIT VISUALIZATION
# ============================================================================

print("\n" + "=" * 90)
print("ANALYSIS 4: GENERATING VISUALIZATION")
print("=" * 90)

# Aggregate data by discount level
viz_data = []
for discount in sorted(df['discount_percent'].unique()):
    subset = df[df['discount_percent'] == discount]
    viz_data.append({
        'discount': discount,
        'revenue': subset['revenue'].sum(),
        'profit': subset['profit'].sum(),
        'transactions': len(subset),
        'avg_profit_per_transaction': subset['profit'].sum() / len(subset)
    })

viz_df = pd.DataFrame(viz_data)

# Create comprehensive visualization
fig = plt.figure(figsize=(16, 12))
fig.suptitle('Retail Pricing Strategy: Discount Impact on Revenue & Profit',
             fontsize=18, fontweight='bold', y=0.995)

# --- PLOT 1: Revenue vs Profit by Discount Level ---
ax1 = plt.subplot(2, 2, 1)
x = np.arange(len(viz_df))
width = 0.35

bars1 = ax1.bar(x - width/2, viz_df['revenue']/1000, width, label='Revenue',
                color='#3498db', alpha=0.8, edgecolor='black', linewidth=1.5)
bars2 = ax1.bar(x + width/2, viz_df['profit']/1000, width, label='Profit',
                color='#2ecc71', alpha=0.8, edgecolor='black', linewidth=1.5)

ax1.set_xlabel('Discount Level (%)', fontweight='bold', fontsize=11)
ax1.set_ylabel('Amount ($1000s)', fontweight='bold', fontsize=11)
ax1.set_title('Revenue vs Profit by Discount Level', fontweight='bold', fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels([f"{int(d)}%" for d in viz_df['discount']])
ax1.legend(fontsize=10, loc='upper right')
ax1.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'${height:.0f}K', ha='center', va='bottom', fontsize=9, fontweight='bold')

# --- PLOT 2: Profit Margin Trend ---
ax2 = plt.subplot(2, 2, 2)
profit_margins = (viz_df['profit'] / viz_df['revenue'] * 100)
line = ax2.plot(viz_df['discount'], profit_margins, marker='o', linewidth=3,
                markersize=10, color='#e74c3c', label='Profit Margin %')
ax2.fill_between(viz_df['discount'], profit_margins, alpha=0.3, color='#e74c3c')

# Highlight sweet spot
sweet_idx = viz_df[viz_df['discount'] == sweet_spot_discount].index[0]
ax2.plot(sweet_spot_discount, profit_margins.iloc[sweet_idx], 'g*',
         markersize=30, label=f'Sweet Spot ({sweet_spot_discount}%)', zorder=5)

ax2.set_xlabel('Discount Level (%)', fontweight='bold', fontsize=11)
ax2.set_ylabel('Profit Margin (%)', fontweight='bold', fontsize=11)
ax2.set_title('Profit Margin Trend Across Discounts', fontweight='bold', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.set_xticks(viz_df['discount'])

# Add value labels
for i, (x, y) in enumerate(zip(viz_df['discount'], profit_margins)):
    ax2.text(x, y + 1, f'{y:.1f}%', ha='center', fontsize=9, fontweight='bold')

# --- PLOT 3: Profit Waterfall by Discount ---
ax3 = plt.subplot(2, 2, 3)
colors_waterfall = ['#2ecc71' if d == sweet_spot_discount else '#3498db'
                    for d in viz_df['discount']]
bars3 = ax3.bar(viz_df['discount'].astype(str), viz_df['profit']/1000,
               color=colors_waterfall, alpha=0.8, edgecolor='black', linewidth=1.5)

ax3.set_xlabel('Discount Level (%)', fontweight='bold', fontsize=11)
ax3.set_ylabel('Total Profit ($1000s)', fontweight='bold', fontsize=11)
ax3.set_title('Total Profit by Discount Level\n(Green = Optimal Discount)',
             fontweight='bold', fontsize=12)
ax3.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels and percentage change
baseline_profit = viz_df.iloc[0]['profit']
for i, (bar, profit) in enumerate(zip(bars3, viz_df['profit'])):
    height = bar.get_height()
    pct_change = ((profit - baseline_profit) / baseline_profit * 100)
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'${height:.0f}K\n({pct_change:+.1f}%)', ha='center', va='bottom',
            fontsize=9, fontweight='bold')

# --- PLOT 4: Transaction Volume & Avg Profit Per Transaction ---
ax4 = plt.subplot(2, 2, 4)
ax4_2 = ax4.twinx()

bar_vol = ax4.bar(viz_df['discount'], viz_df['transactions'], alpha=0.6,
                  color='#9b59b6', label='Transaction Count', edgecolor='black', linewidth=1.5)
line_avg = ax4_2.plot(viz_df['discount'], viz_df['avg_profit_per_transaction'],
                      marker='s', color='#f39c12', linewidth=2.5, markersize=8,
                      label='Avg Profit/Transaction')

ax4.set_xlabel('Discount Level (%)', fontweight='bold', fontsize=11)
ax4.set_ylabel('Transaction Count', fontweight='bold', fontsize=11, color='#9b59b6')
ax4_2.set_ylabel('Avg Profit Per Transaction ($)', fontweight='bold', fontsize=11, color='#f39c12')
ax4.set_title('Transaction Volume & Profitability', fontweight='bold', fontsize=12)
ax4.tick_params(axis='y', labelcolor='#9b59b6')
ax4_2.tick_params(axis='y', labelcolor='#f39c12')
ax4.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels
for bar in bar_vol:
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('discount_analysis_visualization.png', dpi=300, bbox_inches='tight')
print("✓ Visualization saved as 'discount_analysis_visualization.png'")
plt.close()

# ============================================================================
# 6. STRATEGIC RECOMMENDATIONS
# ============================================================================

print("\n" + "=" * 90)
print("STRATEGIC BUSINESS RECOMMENDATIONS")
print("=" * 90)

recommendations = [
    "\n📍 DISCOUNT STRATEGY:",
    f"   1. PRIMARY RECOMMENDATION: Use {sweet_spot_discount}% discount as your standard promotional offer",
    f"      • Maximizes total profit at ${sweet_spot_profit:,.2f}",
    f"      • Balances volume growth with margin preservation",
    f"      • Superior to no-discount strategy by ${sweet_spot_profit - profit_by_discount[0]:,.2f}",

    "\n💰 MARGIN PROTECTION:",
    "   2. Never discount these premium products:",
]

for _, row in product_impact_df.iterrows():
    if row['Best_Discount'] == 0:
        recommendations.append(f"      • {row['Product']} (Best margin: {row['Best_Margin']:.1f}% @ 0% discount)")

recommendations.extend([
    "\n⚠️  HIGH-RISK PRODUCTS:",
    "   3. Use cautious discounting for margin-sensitive items:",
])

for _, row in high_risk.iterrows():
    recommendations.append(f"      • {row['Product']} (Gap: {row['Margin_Gap']:.1f}%) - max {sweet_spot_discount}%")

recommendations.extend([
    "\n📊 VOLUME CONSIDERATION:",
    f"   4. Transaction volume at {sweet_spot_discount}% discount: {viz_df[viz_df['discount']==sweet_spot_discount]['transactions'].values[0]} orders",
    "      • Validates that discount drives incremental volume without excessive margin erosion",

    "\n🎯 SEGMENTATION STRATEGY:",
    "   5. Implement tiered pricing:",
    "      • Premium tier (0% discount): Low-price-sensitive customers, high-value items",
    f"      • Standard tier ({sweet_spot_discount}% discount): Volume-focused customers, competitive positioning",
    "      • Clearance tier (30-50% discount): End-of-life inventory, customer acquisition",

    "\n✅ AVOID DEEP DISCOUNTING:",
    "   6. Higher discounts (30-50%) show diminishing returns:",
    f"      • Excessive margin compression beyond {sweet_spot_discount}% not justified",
    "      • Risk of training customers to expect heavy discounts",
    "      • Devalues brand and product perception"
])

for rec in recommendations:
    print(rec)

print("\n" + "=" * 90)
print("Analysis Complete. Visualization saved.")
print("=" * 90)


RETAIL PRICING STRATEGY ANALYSIS

Dataset Shape: (5000, 9)
Products Analyzed: 5
Discount Levels: [np.int64(0), np.int64(5), np.int64(10), np.int64(20), np.int64(30), np.int64(50)]

ANALYSIS 1: PROFIT MARGIN AT EACH DISCOUNT LEVEL

 Discount % Total Revenue Total Profit Profit Margin % Avg Item Price  Transaction Count
        0%   $797,100.00  $318,840.00          40.00%        $325.41                980
        5%   $850,962.50  $313,512.50          36.84%        $344.57               1031
       10%   $738,720.00  $246,240.00          33.33%        $329.97               1011
       20%   $660,400.00  $165,100.00          25.00%        $340.51                975
       30%   $442,960.00   $63,280.00          14.29%        $325.99                754
       50%   $111,975.00  $-22,395.00         -20.00%        $364.86                249

ANALYSIS 2: SWEET SPOT IDENTIFICATION (MAXIMUM PROFIT DISCOUNT)

✓ OPTIMAL DISCOUNT LEVEL: 0%
  • Total Profit Generated: $318,840.00
  • Total Revenue